In [1]:
import yfinance as yf
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from outils import *
from backtest import *
from metrics import PerformanceMetrics
from sklearn.model_selection import ParameterGrid
import optuna
from functools import partial
import pickle as pkl

/Users/gabriellima/Quant-Challenge/Quant-Challenge/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
start_date = '2015-01-01'
end_date = '2024-11-11'
initial_investment = 1_000_000

period_name = 'presente'
BASE_STOCK_DIR = '../data/stock_data'
BASE_ETF_DIR = '../data/etfs'
DATA_DIR = os.path.join(BASE_STOCK_DIR, period_name)
ADJ_CLOSE_STOCK_FILE = os.path.join(DATA_DIR, 'adj_close_stock_data.parquet.gzip')
ADJ_CLOSE_ETF_FILE = os.path.join(BASE_ETF_DIR, '2015-presente_adj_close.parquet')

adj_close_stock = pd.read_parquet(ADJ_CLOSE_STOCK_FILE).loc[start_date:end_date,:]
adj_close_etf = pd.read_parquet(ADJ_CLOSE_ETF_FILE).loc[start_date:end_date,:]
adj_close_etf.index = adj_close_etf.index.tz_localize(None)

tickers_by_sector_df = pd.read_csv(os.path.join(DATA_DIR, 'tickers_by_sector.csv'))
with open(os.path.join(DATA_DIR, 'tickers_per_rebalance_date.pkl'), 'rb') as f:
    tickers_per_rebalance_date = pkl.load(f)

# Download benchmark data
benchmark_ticker = '^GSPC'
benchmark_data = yf.download(benchmark_ticker, start=start_date, end=end_date)['Adj Close']
benchmark_data.index = benchmark_data.index.tz_localize(None)
benchmark_data.name = 'Index'

[*********************100%***********************]  1 of 1 completed


In [3]:
p = next(iter(tickers_per_rebalance_date.values()))
u = list(tickers_per_rebalance_date.values())[-1]

In [4]:
l = list(set(p) - set(u))

In [5]:
# Defining parameters
fitness_params = {
    'w_return': 1.0,
    'w_momentum': 0.0,
    'w_deviation': -0.5,
    'w_sortino': 0.0,
    'w_cvar': -1.0,
    'w_drawdown': -1.0,
    'w_transaction_cost': -0.5,
    'w_concentration': -0.5,
    'w_portfolio_volatility': -0.0,
    'alpha': 0.95,
    'risk_free_rate': 0.02
}

scaling_factors = load_scaling_factors()

# Run backtest with PSO optimization
portfolio_values, allocations_df = backtest_pso(
    adj_close_stock=adj_close_stock,
    tickers_by_sector_df=tickers_by_sector_df,
    adj_close_etf=adj_close_etf,
    index_data=benchmark_data,
    start_date=start_date,
    end_date=end_date,
    rebalance_period='3ME',
    initial_investment=initial_investment,
)

# Ensure allocations_df index is datetime and sorted
allocations_df.index = pd.to_datetime(allocations_df.index)
allocations_df = allocations_df.sort_index()

# Extend allocations to daily frequency by forward-filling
allocations_daily = allocations_df.reindex(adj_close_stock.index, method='ffill')

# Remove any NaN values
portfolio_values = portfolio_values.dropna()

# Calculate benchmark cumulative returns
benchmark_returns = benchmark_data.pct_change().dropna()
benchmark_cumulative_returns = (1 + benchmark_returns).cumprod() * initial_investment

# Align benchmark data with portfolio dates
benchmark_cumulative_returns = benchmark_cumulative_returns[benchmark_cumulative_returns.index.isin(portfolio_values.index)]

plot_returns(portfolio_values=portfolio_values, benchmark_cumulative_returns=benchmark_cumulative_returns)
plot_portfolio_composition(allocations_daily=allocations_daily)

oi


KeyError: "['CHTR', 'DIS', 'HAS', 'APTV', 'BKNG', 'EL', 'MNST', 'SJM', 'CTRA', 'BKR', 'HAL', 'BK', 'MCO', 'WTW', 'VTRS', 'COR', 'BSX', 'EFX', 'LHX', 'SNA', 'FSLR', 'AKAM', 'CRM', 'ALB', 'CE', 'CBRE', 'CSGP', 'AES', 'NRG', 'NI', 'FXF', 'FXY', 'GLD'] not in index"

In [ ]:
# Calculate daily returns
portfolio_daily_returns = portfolio_values.pct_change().dropna()
benchmark_daily_returns = benchmark_returns[benchmark_returns.index.isin(portfolio_daily_returns.index)]

metrics = PerformanceMetrics()

# Calculate risk metrics for the portfolio
portfolio_annual_return = metrics.calculate_annualized_return(portfolio_daily_returns)
portfolio_annual_volatility = metrics.calculate_annualized_volatility(portfolio_daily_returns)
portfolio_sharpe_ratio = metrics.calculate_sharpe_ratio(portfolio_daily_returns)
portfolio_max_drawdown = metrics.calculate_max_drawdown(portfolio_values)

# Calculate risk metrics for the benchmark
benchmark_annual_return = metrics.calculate_annualized_return(benchmark_daily_returns)
benchmark_annual_volatility = metrics.calculate_annualized_volatility(benchmark_daily_returns)
benchmark_sharpe_ratio = metrics.calculate_sharpe_ratio(benchmark_daily_returns)
benchmark_max_drawdown = metrics.calculate_max_drawdown(benchmark_cumulative_returns)

# Print risk metrics
print("Risk Metrics:")
print("\nOptimized Portfolio:")
print(f"Annualized Return: {portfolio_annual_return:.2%}")
print(f"Annualized Volatility: {portfolio_annual_volatility:.2%}")
print(f"Sharpe Ratio: {portfolio_sharpe_ratio:.2f}")
print(f"Maximum Drawdown: {portfolio_max_drawdown:.2%}")

print("\nS&P 500 Benchmark:")
print(f"Annualized Return: {benchmark_annual_return:.2%}")
print(f"Annualized Volatility: {benchmark_annual_volatility:.2%}")
print(f"Sharpe Ratio: {benchmark_sharpe_ratio:.2f}")
print(f"Maximum Drawdown: {benchmark_max_drawdown:.2%}")